In [2]:
# Import data manipulation libraries
import pandas as pd
import numpy as np

In [3]:
# Load preprocessed dataset created in previous notebook

df = pd.read_parquet(
    "../data/processed/preprocessed_data.parquet"
)

In [4]:
# Display first rows

df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,event_type_1,event_name_2,event_type_2,snap_CA,snap_TX,snap_WI,sell_price,rolling_mean_28,rolling_std_28,is_outlier
0,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_1,3,2011-01-29,11101,...,2,3,1,0,0,0,2.0,NaN,NaN,False
1,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_2,0,2011-01-30,11101,...,2,3,1,0,0,0,2.0,NaN,NaN,False
2,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_3,0,2011-01-31,11101,...,2,3,1,0,0,0,2.0,NaN,NaN,False
3,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_4,1,2011-02-01,11101,...,2,3,1,1,1,0,2.0,NaN,NaN,False
4,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_5,4,2011-02-02,11101,...,2,3,1,1,0,1,2.0,NaN,NaN,False


In [5]:
# Check dataset structure

df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 58327370 entries, 0 to 58327369
Data columns (total 26 columns):
 #   Column           Dtype         
---  ------           -----         
 0   id               object        
 1   item_id          int64         
 2   dept_id          int64         
 3   cat_id           int64         
 4   store_id         int64         
 5   state_id         int64         
 6   day              object        
 7   sales            int64         
 8   date             datetime64[ns]
 9   wm_yr_wk         int64         
 10  weekday          object        
 11  wday             int64         
 12  month            int64         
 13  year             int64         
 14  d                object        
 15  event_name_1     int64         
 16  event_type_1     int64         
 17  event_name_2     int64         
 18  event_type_2     int64         
 19  snap_CA          int64         
 20  snap_TX          int64         
 21  snap_WI          int64       

In [6]:
# Sort dataset chronologically by product and date
# Required before creating lag and rolling features

df = df.sort_values(
    by=["id", "date"]
)

LAG FEATURES

In [7]:
# Create 7-day lag feature
# Represents sales from 7 days ago

df["lag_7"] = (
    df.groupby("id")["sales"]
    .shift(7)
)

# Create 14-day lag feature

df["lag_14"] = (
    df.groupby("id")["sales"]
    .shift(14)
)

# Create 28-day lag feature

df["lag_28"] = (
    df.groupby("id")["sales"]
    .shift(28)
)

ROLLING FEATURES

In [8]:
# Create rolling 7-day average sales
# Uses past 7 days only

df["rolling_mean_7"] = (
    df.groupby("id")["sales"]
    .transform(
        lambda x: x.shift(7).rolling(7).mean()
    )
)

# Create rolling 28-day average sales

df["rolling_mean_28"] = (
    df.groupby("id")["sales"]
    .transform(
        lambda x: x.shift(28).rolling(28).mean()
    )
)

In [9]:
# Create rolling 7-day standard deviation
# Measures recent sales volatility

df["rolling_std_7"] = (
    df.groupby("id")["sales"]
    .transform(
        lambda x: x.shift(7).rolling(7).std()
    )
)

TEMPORAL FEATURES

In [10]:
# Extract day of week
# Monday=0, Sunday=6

df["day_of_week"] = df["date"].dt.weekday

# Extract week number

df["week_of_year"] = (
    df["date"].dt.isocalendar().week
)

# Extract month

df["month"] = df["date"].dt.month

# Extract year

df["year"] = df["date"].dt.year

# Create weekend indicator

df["is_weekend"] = (
    df["day_of_week"]
    .isin([5, 6])
    .astype(int)
)

EVENT FEATURES

In [11]:
# Create SNAP indicator
# SNAP days influence retail demand

df["is_snap_day"] = (
    (
        (df["state_id"] == 0) & (df["snap_CA"] == 1)
    ) |
    (
        (df["state_id"] == 1) & (df["snap_TX"] == 1)
    ) |
    (
        (df["state_id"] == 2) & (df["snap_WI"] == 1)
    )
).astype(int)

In [12]:
# Create sporting event indicator

df["is_sporting_event"] = (
    df["event_type_1"] == 1
).astype(int)

# Create cultural event indicator

df["is_cultural_event"] = (
    df["event_type_1"] == 0
).astype(int)

# Create national event indicator

df["is_national_event"] = (
    df["event_type_1"] == 2
).astype(int)

# Create religious event indicator

df["is_religious_event"] = (
    df["event_type_1"] == 3
).astype(int)

PRICE FEATURES

In [13]:
# Calculate week-to-week price change

df["price_change"] = (
    df.groupby(["id"])["sell_price"]
    .diff()
)

In [14]:
# Remove rows with missing feature values

df = df.dropna()

In [15]:
# Display first rows

df.head()

,id,item_id,dept_id,cat_id,store_id,state_id,day,sales,date,wm_yr_wk,...,rolling_std_7,day_of_week,week_of_year,is_weekend,is_snap_day,is_sporting_event,is_cultural_event,is_national_event,is_religious_event,price_change
55,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_56,1,2011-03-25,11108,...,1.825742,4,12,0,0,0,0,1,0,0.0
56,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_57,2,2011-03-26,11109,...,1.825742,5,12,1,0,0,0,1,0,0.0
57,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_58,2,2011-03-27,11109,...,0.534522,6,12,1,0,0,0,1,0,0.0
58,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_59,0,2011-03-28,11109,...,0.534522,0,13,0,0,0,0,1,0,0.0
59,FOODS_1_001_CA_1_validation,0,0,0,0,0,d_60,1,2011-03-29,11109,...,0.487950,1,13,0,0,0,0,1,0,0.0


In [16]:
# Check dataset structure

df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 56650420 entries, 55 to 58327369
Data columns (total 40 columns):
 #   Column              Dtype         
---  ------              -----         
 0   id                  object        
 1   item_id             int64         
 2   dept_id             int64         
 3   cat_id              int64         
 4   store_id            int64         
 5   state_id            int64         
 6   day                 object        
 7   sales               int64         
 8   date                datetime64[ns]
 9   wm_yr_wk            int64         
 10  weekday             object        
 11  wday                int64         
 12  month               int32         
 13  year                int32         
 14  d                   object        
 15  event_name_1        int64         
 16  event_type_1        int64         
 17  event_name_2        int64         
 18  event_type_2        int64         
 19  snap_CA             int64         
 20  snap

In [17]:
# ── Event proximity features (methodology §3.5.3 / §3.5.4) ─────────────────
# An "event" is any calendar day with event_name_1 or event_name_2 ≠ "No Event".
# After label encoding, "No Event" maps to a specific integer. Detect it.

# event_name columns are label-encoded; the modal value is "No Event"
no_event_code_1 = df["event_name_1"].mode().iloc[0]
no_event_code_2 = df["event_name_2"].mode().iloc[0]

df["_is_event_day"] = (
    (df["event_name_1"] != no_event_code_1)
    | (df["event_name_2"] != no_event_code_2)
).astype("int8")

# Compute per-date event flag (events are calendar-wide, not per-series)
date_events = (
    df.groupby("date")["_is_event_day"].max().reset_index()
      .sort_values("date").reset_index(drop=True)
)

# days_since_last_event: running count since most recent event day
last_idx = -1
since = []
for i, ev in enumerate(date_events["_is_event_day"].values):
    if ev == 1:
        last_idx = i
    since.append(i - last_idx if last_idx >= 0 else 999)
date_events["days_since_last_event"] = since

# days_to_next_event: count to upcoming event day
next_idx = len(date_events)
to_next = [0] * len(date_events)
for i in range(len(date_events) - 1, -1, -1):
    if date_events["_is_event_day"].iat[i] == 1:
        next_idx = i
    to_next[i] = (next_idx - i) if next_idx < len(date_events) else 999
date_events["days_to_next_event"] = to_next

df = df.merge(
    date_events[["date", "days_since_last_event", "days_to_next_event"]],
    on="date", how="left",
)
df = df.drop(columns=["_is_event_day"])

print(df[["date", "days_since_last_event", "days_to_next_event"]].head(10))
print("nulls:", df[["days_since_last_event", "days_to_next_event"]].isnull().sum().to_dict())

        date  days_since_last_event  days_to_next_event
0 2011-03-25                    999                  30
1 2011-03-26                    999                  29
2 2011-03-27                    999                  28
3 2011-03-28                    999                  27
4 2011-03-29                    999                  26
5 2011-03-30                    999                  25
6 2011-03-31                    999                  24
7 2011-04-01                    999                  23
8 2011-04-02                    999                  22
9 2011-04-03                    999                  21
nulls: {'days_since_last_event': 0, 'days_to_next_event': 0}


In [18]:
# Save engineered feature dataset

df.to_parquet(
    "../data/processed/feature_engineered_data.parquet",
    index=False
)